# Amazon ML Challenge 2026 — Business Entity Resolution

**normalise → GPU blocking (char-3-gram sketch + FAISS) → pairwise features → PyTorch MLP matcher (GPU) → threshold tuned for macro F0.5 → submission**

Run from `code/business_entity_resolution/`; reuses the modules in `src/`. On Kaggle use `er_pipeline_kaggle.ipynb`, which writes `src/` itself — attach only the dataset.

In [14]:
# Local: run from code/business_entity_resolution/. Kaggle: use er_pipeline_kaggle.ipynb (writes src/ itself).
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    !pip install -q polars==1.44.2 faiss-gpu-cu12 anyascii rapidfuzz sparse-dot-topn
    Path("src").mkdir(exist_ok=True)

## 0. Source modules
Written to `src/` so the rest of the notebook imports them as on a local run.

In [15]:
%%writefile src/normalize.py
"""Text normalisation shared by every stage of the pipeline.

Country-agnostic: every script is transliterated to ASCII with anyascii, so
the same code handles US, India (incl. Indic-script names) and unseen France.
"""
import re

from anyascii import anyascii

# Placeholder values that appear inside addresses and carry no information.
PLACEHOLDER_RE = re.compile(r"<\s*null\s*>|\bn/?a\b|\bnull\b|\bnone\b", re.IGNORECASE)
NON_ALNUM_RE = re.compile(r"[^a-z0-9]+")

# Common transliteration outputs of Indic-script legal suffixes -> English form.
# Seeded from a handful of anyascii outputs; extend by mining training pairs.
TRANSLIT_FIXES = {
    "praibhet": "private", "praivet": "private", "piraivet": "private",
    "praivhet": "private", "praiveta": "private", "pryvett": "private",
    "limitet": "limited", "limitedd": "limited", "limitadd": "limited",
    "elelpi": "llp", "elelpii": "llp",
}

# Legal-form tokens, mapped to one canonical spelling.
LEGAL_CANON = {
    "pvt": "private", "pvtltd": "private limited",
    "ltd": "limited", "ltda": "limited",
    "corp": "corporation", "inc": "incorporated", "incorp": "incorporated",
    "co": "company", "cos": "company",
    "llc": "llc", "l l c": "llc", "pllc": "pllc", "llp": "llp", "lp": "lp",
    "sarl": "sarl", "sas": "sas", "sasu": "sasu", "sa": "sa", "eurl": "eurl", "sci": "sci",
}
LEGAL_TOKENS = set(LEGAL_CANON.values()) | {"private", "limited", "company", "corporation",
                                             "incorporated", "ei", "snc"}

# Address abbreviations (English and French street types).
ADDR_CANON = {
    "st": "street", "str": "street", "rd": "road", "ave": "avenue", "av": "avenue",
    "blvd": "boulevard", "bd": "boulevard", "dr": "drive", "ln": "lane", "ct": "court",
    "pl": "place", "pkwy": "parkway", "hwy": "highway", "cir": "circle", "ter": "terrace",
    "sq": "square", "apt": "apartment", "ste": "suite", "fl": "floor", "flr": "floor",
    "no": "number", "nr": "near", "opp": "opposite", "mkt": "market",
    "n": "north", "s": "south", "e": "east", "w": "west",
    "r": "rue", "ch": "chemin", "rte": "route", "imp": "impasse",
}


def _basic(text: str | None) -> str:
    if not text:
        return ""
    text = anyascii(text).lower()
    text = text.replace("&", " and ").replace("+", " and ")
    return NON_ALNUM_RE.sub(" ", text).strip()


def norm_name(text: str | None) -> str:
    tokens = _basic(text).split()
    out = []
    for t in tokens:
        t = TRANSLIT_FIXES.get(t, t)
        out.append(LEGAL_CANON.get(t, t))
    return " ".join(out)


def core_name(normed_name: str) -> str:
    """Name with legal-form tokens removed ("blue strategy llc" -> "blue strategy")."""
    kept = [t for t in normed_name.split() if t not in LEGAL_TOKENS]
    return " ".join(kept) if kept else normed_name


def norm_addr(text: str | None) -> str:
    if not text:
        return ""
    text = PLACEHOLDER_RE.sub(" ", text)
    return " ".join(ADDR_CANON.get(t, t) for t in _basic(text).split())

Overwriting src/normalize.py


In [16]:
%%writefile src/prepare.py
"""Normalise every source file once and cache it as parquet.

Usage: python src/prepare.py --data-dir <student_resource/dataset> --cache-dir <cache>
Output: <cache>/{train,test}_source{1,2,3}.parquet with columns
        entity_id, country, name, core, addr (all normalised), is_translit
"""
import argparse
from multiprocessing import Pool
from pathlib import Path

import polars as pl

from normalize import core_name, norm_addr, norm_name

CHUNK = 100_000


def _norm_chunk(rows):
    names, addrs = rows
    n = [norm_name(x) for x in names]
    return n, [core_name(x) for x in n], [norm_addr(x) for x in addrs]


def read_source(path: Path) -> pl.DataFrame:
    return pl.read_csv(path, separator="\t", quote_char=None, infer_schema=False)


def prepare_file(src: Path, dst: Path, pool: Pool) -> None:
    df = read_source(src)
    chunks = ((df["business_name"][i:i + CHUNK].to_list(), df["business_address"][i:i + CHUNK].to_list())
              for i in range(0, df.height, CHUNK))
    parts = [pl.DataFrame({"name": cn, "core": cc, "addr": ca}, schema={k: pl.String for k in ("name", "core", "addr")})
             for cn, cc, ca in pool.imap(_norm_chunk, chunks)]
    normed = pl.concat(parts)
    out = df.select(
        "entity_id", "country",
        # Name was not in Latin script originally (e.g. Tamil / Devanagari).
        pl.col("business_name").str.contains(r"[\p{L}&&[^\p{Latin}]]").alias("is_translit"),
    ).hstack(normed)
    out.write_parquet(dst)
    print(f"{src.name}: {out.height:,} rows -> {dst}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", type=Path, required=True)
    ap.add_argument("--cache-dir", type=Path, required=True)
    args = ap.parse_args()
    args.cache_dir.mkdir(parents=True, exist_ok=True)
    with Pool(6) as pool:
        for split in ("train", "test"):
            for s in (1, 2, 3):
                name = f"{split}_source{s}"
                dst = args.cache_dir / f"{name}.parquet"
                if not dst.exists():
                    prepare_file(args.data_dir / split / f"{name}.tsv", dst, pool)


if __name__ == "__main__":
    main()

Overwriting src/prepare.py


In [17]:
%%writefile src/sample.py
"""Build a small development sample of the training data.

Keeps a fraction of Source 1 entities together with ALL their true S2/S3 matches,
plus the same fraction of the unmatched S2/S3 records (the decoys). Every real
business is sampled at the same rate, so the sample looks like a smaller world.

Caveat: competition for each S1 is ~1/frac of the full data, so recall@K on the
sample is optimistic; it roughly corresponds to recall@(K/frac) on the full set.

Usage: python src/sample.py --data-dir <dataset> --cache-dir <cache> --frac 0.1
"""
import argparse
from pathlib import Path

import polars as pl

from prepare import read_source


def load_links(gt_path: Path) -> pl.DataFrame:
    """Ground truth as one row per (source1_entity_id, entity_id) true pair."""
    gt = read_source(gt_path)
    return (gt.with_columns(pl.col("matched_entity_ids").str.split(","))
              .explode("matched_entity_ids", empty_as_null=True)
              .drop_nulls("matched_entity_ids")
              .rename({"matched_entity_ids": "entity_id"}))


def keep(col: str, frac: float, seed: int) -> pl.Expr:
    return (pl.col(col).hash(seed) % 10_000) < int(frac * 10_000)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", type=Path, required=True)
    ap.add_argument("--cache-dir", type=Path, required=True)
    ap.add_argument("--frac", type=float, default=0.1)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    c = args.cache_dir

    links = load_links(args.data_dir / "train" / "train_ground_truth.tsv")
    s1 = pl.read_parquet(c / "train_source1.parquet").filter(keep("entity_id", args.frac, args.seed))
    pool = pl.concat([pl.read_parquet(c / "train_source2.parquet"), pl.read_parquet(c / "train_source3.parquet")])

    is_decoy = ~pl.col("entity_id").is_in(links["entity_id"].implode())  # matches no S1 at all
    links = links.filter(pl.col("source1_entity_id").is_in(s1["entity_id"].implode()))
    pool = pool.filter(pl.col("entity_id").is_in(links["entity_id"].implode())
                       | (is_decoy & keep("entity_id", args.frac, args.seed)))

    tag = f"dev{int(args.frac * 100)}"
    s1.write_parquet(c / f"{tag}_source1.parquet")
    pool.write_parquet(c / f"{tag}_pool.parquet")
    links.write_parquet(c / f"{tag}_links.parquet")
    print(f"{tag}: S1 {s1.height:,}  pool {pool.height:,}  true pairs {links.height:,}")
    print(s1.group_by("country").len(), pool.group_by("country").len())


if __name__ == "__main__":
    main()

Overwriting src/sample.py


In [18]:
%%writefile src/candidates.py
"""Candidate generation (blocking): TF-IDF nearest-neighbour search.

For every Source 1 record we search Source 2/3 records of the SAME country and keep
the top-K most similar under several "views" of the record:

    name : character 3-grams of the core name (legal suffixes removed)
    addr : character 3-grams of the address  (rescues transliterated / duplicate names)
    full : character 3-grams of name + address

Plus a reverse pass: for every S2/S3 record, its top few S1 records under the
"full" view. This helps S1 entities with very common names ("Meridian"), where
the forward top-K fills up with lookalikes.

The union of all views is the candidate set. Each view's cosine similarity is
kept as a column so the matching model can use it as a feature.

Usage:
  python src/candidates.py --cache-dir <cache> --split dev10              # sample
  python src/candidates.py --cache-dir <cache> --split test \
         --out-tsv output/candidate_pairs.tsv                             # submission
"""
import argparse
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn

S1_KEY, CAND_KEY = "source1_entity_id", "entity_id"


@dataclass
class View:
    name: str
    top_k: int            # forward: candidates per S1 record
    reverse_k: int = 0    # reverse: S1 records per S2/S3 record (0 = off)
    min_sim: float = 0.1  # drop neighbours below this cosine similarity


VIEWS = [
    View("name", top_k=30),
    View("addr", top_k=30),
    View("full", top_k=30, reverse_k=3),
]


def view_text(df: pl.DataFrame, view: str) -> list[str]:
    if view == "name":
        return df["core"].to_list()
    if view == "addr":
        return df["addr"].to_list()
    return (df["core"] + " " + df["addr"]).to_list()


def top_neighbours(q, p, k: int, min_sim: float, n_threads: int, chunk: int, device: str = "cpu"):
    """Row i of q -> its top-k rows of p by cosine (rows are L2-normalised)."""
    if device != "cpu":
        return top_neighbours_gpu(q, p, k, min_sim, device)
    pt = p.T.tocsr()
    rows, cols, sims = [], [], []
    for start in range(0, q.shape[0], chunk):
        m = sp_matmul_topn(q[start:start + chunk], pt, top_n=k, threshold=min_sim,
                           n_threads=n_threads).tocoo()
        rows.append(m.row + start); cols.append(m.col); sims.append(m.data)
    return np.concatenate(rows), np.concatenate(cols), np.concatenate(sims)


def top_neighbours_gpu(q, p, k: int, min_sim: float, device: str, budget: float = 2.5e8):
    """GPU version: p stays sparse on the GPU, queries go in dense batches.

    Each batch computes p @ batch.T (n_p x b similarities) and takes top-k per
    column. `budget` caps the floats held per batch (~1 GB at the default).
    """
    import torch

    p = p.tocsr()
    P = torch.sparse_csr_tensor(torch.from_numpy(p.indptr.astype(np.int32)),
                                torch.from_numpy(p.indices.astype(np.int32)),
                                torch.from_numpy(p.data), size=p.shape, device=device)
    k = min(k, p.shape[0])
    b = max(1, int(min(budget / p.shape[0], budget / p.shape[1])))
    q = q.tocsr()
    rows, cols, sims = [], [], []
    for start in range(0, q.shape[0], b):
        batch = torch.from_numpy(q[start:start + b].T.toarray()).to(device)  # V x b
        vals, idx = torch.topk(torch.sparse.mm(P, batch), k, dim=0)          # k x b
        keep = vals >= min_sim
        rows.append((keep.nonzero()[:, 1] + start).cpu().numpy())
        cols.append(idx[keep].cpu().numpy())
        sims.append(vals[keep].cpu().numpy())
    del P
    torch.cuda.empty_cache()
    return np.concatenate(rows), np.concatenate(cols), np.concatenate(sims)


def search_country(s1: pl.DataFrame, pool: pl.DataFrame, views: list[View],
                   n_threads: int, chunk: int, device: str) -> pl.DataFrame:
    s1_ids, pool_ids = s1["entity_id"].to_numpy(), pool["entity_id"].to_numpy()
    found = []
    for v in views:
        q_text, p_text = view_text(s1, v.name), view_text(pool, v.name)
        vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3), min_df=2, max_df=0.05,
                              dtype=np.float32, sublinear_tf=True)
        vec.fit(q_text + p_text)  # unsupervised: fitting on test text is fine
        q, p = vec.transform(q_text), vec.transform(p_text)

        r, c, s = top_neighbours(q, p, v.top_k, v.min_sim, n_threads, chunk, device)
        pairs = [(r, c, s)]
        if v.reverse_k:
            pr, qr, s_rev = top_neighbours(p, q, v.reverse_k, v.min_sim, n_threads, chunk, device)
            pairs.append((qr, pr, s_rev))
        for qi, pi, sim in pairs:
            found.append(pl.DataFrame({S1_KEY: s1_ids[qi], CAND_KEY: pool_ids[pi],
                                       "view": v.name, "sim": sim.astype(np.float32)}))
    return pl.concat(found)


def generate_candidates(s1: pl.DataFrame, pool: pl.DataFrame, views: list[View] = VIEWS,
                        n_threads: int = 8, chunk: int = 20_000, device: str = "cpu") -> pl.DataFrame:
    """Long table of (S1 id, S2/S3 id, sim_<view>...) over all countries.

    Countries are taken from the data (open set), never hard-coded.
    """
    parts = []
    for country in s1["country"].unique().sort().to_list():
        s1_c = s1.filter(pl.col("country") == country)
        pool_c = pool.filter(pl.col("country") == country)
        if pool_c.height == 0:
            continue
        parts.append(search_country(s1_c, pool_c, views, n_threads, chunk, device))
        print(f"  {country}: {s1_c.height:,} S1 vs {pool_c.height:,} S2/S3")
    return to_wide(pl.concat(parts), views)


def to_wide(long: pl.DataFrame, views: list[View]) -> pl.DataFrame:
    """One row per pair; a pair found by several views gets one sim column per view."""
    wide = long.pivot(on="view", index=[S1_KEY, CAND_KEY], values="sim", aggregate_function="max")
    return wide.rename({v.name: f"sim_{v.name}" for v in views if v.name in wide.columns})


def write_candidate_file(cands: pl.DataFrame, s1_ids: pl.Series, path: Path) -> None:
    """candidate_pairs.tsv: exactly one row per S1 entity, empty list if no candidates."""
    lists = cands.group_by(S1_KEY).agg(pl.col(CAND_KEY).unique().sort().str.join(",").alias("candidate_entity_ids"))
    out = (pl.DataFrame({S1_KEY: s1_ids}).join(lists, on=S1_KEY, how="left")
             .with_columns(pl.col("candidate_entity_ids").fill_null("")))
    path.parent.mkdir(parents=True, exist_ok=True)
    out.write_csv(path, separator="\t", quote_style="never")


def load_split(cache: Path, split: str) -> tuple[pl.DataFrame, pl.DataFrame]:
    if split.startswith("dev"):
        return pl.read_parquet(cache / f"{split}_source1.parquet"), pl.read_parquet(cache / f"{split}_pool.parquet")
    s1 = pl.read_parquet(cache / f"{split}_source1.parquet")
    pool = pl.concat([pl.read_parquet(cache / f"{split}_source2.parquet"),
                      pl.read_parquet(cache / f"{split}_source3.parquet")])
    return s1, pool


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache-dir", type=Path, required=True)
    ap.add_argument("--split", default="dev10", help="dev10 | train | test")
    ap.add_argument("--threads", type=int, default=8)
    ap.add_argument("--device", default="cpu", help="cpu | cuda")
    ap.add_argument("--out-tsv", type=Path, help="also write candidate_pairs.tsv here")
    args = ap.parse_args()

    s1, pool = load_split(args.cache_dir, args.split)
    cands = generate_candidates(s1, pool, n_threads=args.threads, device=args.device)
    out = args.cache_dir / f"{args.split}_candidates.parquet"
    cands.write_parquet(out)
    print(f"{cands.height:,} candidate pairs, {cands.height / s1.height:.1f} per S1 -> {out}")
    if args.out_tsv:
        write_candidate_file(cands, s1["entity_id"], args.out_tsv)


if __name__ == "__main__":
    main()

Overwriting src/candidates.py


In [19]:
%%writefile src/eval_blocking.py
"""Measure how good a candidate set is, using the training labels.

Reports:
  pair recall      share of true (S1, S2/S3) pairs that are in the candidate set
  ceiling F0.5     the score a PERFECT matching model would get with these candidates
                   (it keeps exactly the true pairs it was given, nothing else)
  cands per S1     how much work the matching model has to do
  recall@k         per view, using only its top-k neighbours (helps choose K)

Usage: python src/eval_blocking.py --cache-dir <cache> --split dev10
"""
import argparse
from pathlib import Path

import polars as pl

from candidates import CAND_KEY, S1_KEY


def f05(p: pl.Expr, r: pl.Expr) -> pl.Expr:
    return (1.25 * p * r / (0.25 * p + r)).fill_nan(0.0)


def evaluate(cands: pl.DataFrame, links: pl.DataFrame, s1: pl.DataFrame) -> None:
    pool_meta = cands.select(S1_KEY, CAND_KEY).with_columns(hit=pl.lit(True))
    lk = links.join(pool_meta, on=[S1_KEY, CAND_KEY], how="left").with_columns(pl.col("hit").fill_null(False))
    print(f"pair recall: {lk['hit'].mean():.4f}  ({lk['hit'].sum():,} / {lk.height:,})")

    # Ceiling macro F0.5: precision 1, recall = found / true; singletons score 1.
    per = (s1.select(pl.col("entity_id").alias(S1_KEY), "country")
             .join(lk.group_by(S1_KEY).agg(pl.len().alias("n_true"), pl.col("hit").sum().alias("n_found")),
                   on=S1_KEY, how="left")
             .with_columns(pl.col("n_true", "n_found").fill_null(0))
             .with_columns(pl.when(pl.col("n_true") == 0).then(1.0)
                             .otherwise(f05(pl.lit(1.0), pl.col("n_found") / pl.col("n_true"))).alias("f05")))
    n_c = cands.group_by(S1_KEY).len()
    print(f"ceiling F0.5: {per['f05'].mean():.4f}   cands per S1: {cands.height / s1.height:.1f} "
          f"(max {n_c['len'].max() if n_c.height else 0})")
    print(per.group_by("country").agg(pl.col("f05").mean().alias("ceiling_f05"),
                                      (pl.col("n_found").sum() / pl.col("n_true").sum()).alias("pair_recall")))

    # Recall on matches whose name was in a non-Latin script.
    if "is_translit" in links.columns:
        print(lk.group_by("is_translit").agg(pl.col("hit").mean().alias("pair_recall"), pl.len()))

    # Per-view recall@k: rank each view's neighbours by similarity within an S1.
    for col in [c for c in cands.columns if c.startswith("sim_")]:
        ranked = (cands.drop_nulls(col)
                       .with_columns(pl.col(col).rank("ordinal", descending=True).over(S1_KEY).alias("rk"))
                       .select(S1_KEY, CAND_KEY, "rk"))
        j = links.join(ranked, on=[S1_KEY, CAND_KEY], how="left")
        at = {k: (j["rk"] <= k).sum() / links.height for k in (1, 5, 10, 20, 30)}
        print(f"  {col:10s} " + "  ".join(f"@{k}={v:.3f}" for k, v in at.items()))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache-dir", type=Path, required=True)
    ap.add_argument("--split", default="dev10")
    args = ap.parse_args()
    c = args.cache_dir

    cands = pl.read_parquet(c / f"{args.split}_candidates.parquet")
    s1 = pl.read_parquet(c / f"{args.split}_source1.parquet")
    links = pl.read_parquet(c / f"{args.split}_links.parquet")
    pool = pl.read_parquet(c / f"{args.split}_pool.parquet", columns=["entity_id", "is_translit"])
    evaluate(cands, links.join(pool, on=CAND_KEY, how="left"), s1)


if __name__ == "__main__":
    main()

Overwriting src/eval_blocking.py


In [20]:
%%writefile src/ann_blocking.py
"""Fast candidate generation (blocking) on the GPU with FAISS.

Same idea as candidates.py -- per country and per view, keep each S1 record's
top-K neighbours by TF-IDF char-3-gram cosine -- but approximate, so it scales
to the full test set (exact search there is ~10^13 pair scores per view):

  1. Char-3-gram TF-IDF, count-sketched on the GPU: every record -> a `dim`-d unit vector
  2. FAISS IVF index on the GPU: each query scans only its `nprobe` nearest clusters
  3. The dense cosine is kept as the view's similarity feature (sim_<view>)

Pairs are kept as integer row numbers into the S1 / pool frames (not string IDs)
and sorted by S1 row, which keeps memory ~5x lower and lets later stages work
in S1 chunks.
"""
import time
from dataclasses import dataclass

import faiss
import numpy as np
import polars as pl
import torch
import torch.nn.functional as F

from candidates import CAND_KEY, S1_KEY, VIEWS, View, view_text


@dataclass
class AnnConfig:
    dim: int = 256              # embedding size (count-sketch dims)
    fit_sample: int = 300_000   # records used to train the IVF centroids
    nprobe: int = 32            # IVF clusters scanned per query (higher = better recall, slower)
    exact_below: int = 100_000  # smaller pools use exact (flat) search
    chunk: int = 100_000


class GramSketch:
    """Char-3-gram TF-IDF vectors, count-sketched to `dim` dims, computed on the GPU.

    Text is ASCII after normalisation, so a 3-gram is a 21-bit code (no hashing
    collisions). Each code gets an IDF weight and is added, with a random sign, to
    one of `dim` output dims -- a random projection that approximately preserves
    cosine similarity, including for rare 3-grams (unlike SVD).
    """
    CODES = 1 << 21

    def __init__(self, dim: int = 256, max_len: int = 96, min_df: int = 2, max_df: float = 0.05,
                 chunk: int = 100_000, device: str = "cuda", seed: int = 0):
        g = torch.Generator().manual_seed(seed)
        self.dim_of = torch.randint(0, dim, (self.CODES,), generator=g).to(device)
        self.sign_of = (torch.randint(0, 2, (self.CODES,), generator=g) * 2 - 1).float().to(device)
        self.dim, self.max_len, self.min_df, self.max_df = dim, max_len, min_df, max_df
        self.chunk, self.device = chunk, device

    def _codes(self, texts: list[str]) -> tuple[torch.Tensor, torch.Tensor]:
        """-> (codes, valid) of shape (n, max_len); text is padded with a space on both ends."""
        raw = np.array([t.encode("ascii", "ignore")[: self.max_len - 2] for t in texts], dtype=f"S{self.max_len - 2}")
        x = torch.from_numpy(raw.view(np.uint8).reshape(len(texts), -1).astype(np.int64)).to(self.device)
        lens = (x != 0).sum(1, keepdim=True)
        x = F.pad(x, (1, 2), value=0)
        x[:, 0] = 32
        x.scatter_(1, lens + 1, 32)
        codes = (x[:, :-2] << 14) | (x[:, 1:-1] << 7) | x[:, 2:]
        valid = torch.arange(codes.shape[1], device=self.device) < lens
        return codes & (self.CODES - 1), valid

    def fit(self, texts: list[str]) -> "GramSketch":
        df = torch.zeros(self.CODES, device=self.device)
        for i in range(0, len(texts), self.chunk):
            codes, valid = self._codes(texts[i:i + self.chunk])
            df += torch.bincount(codes[valid], minlength=self.CODES)
        n = len(texts)
        self.idf = torch.log((1 + n) / (1 + df)) + 1
        self.idf[(df < self.min_df) | (df > self.max_df * n)] = 0
        return self

    @torch.no_grad()
    def transform(self, texts: list[str]) -> np.ndarray:
        """Texts -> (n, dim) float16 unit vectors."""
        out = []
        for i in range(0, len(texts), self.chunk):
            codes, valid = self._codes(texts[i:i + self.chunk])
            w = self.idf[codes] * self.sign_of[codes] * valid
            v = torch.zeros(len(codes), self.dim, device=self.device).scatter_add_(1, self.dim_of[codes], w)
            out.append(F.normalize(v, dim=1).half().cpu().numpy())
        return np.vstack(out)


def knn(base: np.ndarray, queries: np.ndarray, k: int, cfg: AnnConfig, res) -> tuple[np.ndarray, np.ndarray]:
    """Top-k inner product of each query against `base`, on the GPU. Returns (sims, ids)."""
    n, d = base.shape
    if n < cfg.exact_below:
        cpu_index = faiss.IndexFlatIP(d)
    else:
        cpu_index = faiss.index_factory(d, f"IVF{int(4 * np.sqrt(n))},SQfp16", faiss.METRIC_INNER_PRODUCT)
    index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
    if not index.is_trained:
        sample = np.random.default_rng(0).choice(n, min(cfg.fit_sample, n), replace=False)
        index.train(base[sample].astype(np.float32))
    for i in range(0, n, cfg.chunk):
        index.add(base[i:i + cfg.chunk].astype(np.float32))
    if hasattr(index, "nprobe"):
        index.nprobe = cfg.nprobe
    k = min(k, n)
    step = max(1_000, (1 << 23) // (k * cfg.nprobe))  # FAISS temp memory ~ step * k * nprobe * 12 bytes (~100 MB)
    out = [index.search(queries[i:i + step].astype(np.float32), k) for i in range(0, len(queries), step)]
    del index
    return np.vstack([o[0] for o in out]), np.vstack([o[1] for o in out])


def search_country(s1: pl.DataFrame, pool: pl.DataFrame, views: list[View], cfg: AnnConfig, res):
    """-> (s1_idx, pool_idx, sims[n_pairs, n_views]) with indices local to s1 / pool, sorted by s1_idx."""
    n_pool = pool.height
    keys, view_ids, sims = [], [], []

    def add(qi, pi, sim, j, min_sim):
        keep = (pi >= 0) & (qi >= 0) & (sim >= min_sim)
        keys.append(qi[keep].astype(np.int64) * n_pool + pi[keep])
        view_ids.append(np.full(keep.sum(), j, np.int8))
        sims.append(sim[keep])

    for j, v in enumerate(views):
        t0 = time.time()
        q_text, p_text = view_text(s1, v.name), view_text(pool, v.name)
        enc = GramSketch(cfg.dim).fit(p_text)
        q, p = enc.transform(q_text), enc.transform(p_text)
        del enc
        torch.cuda.empty_cache()  # hand GPU memory back to FAISS
        t1 = time.time()
        s, nb = knn(p, q, v.top_k, cfg, res)                        # S1 -> pool
        add(np.arange(len(q)).repeat(nb.shape[1]), nb.ravel(), s.ravel(), j, v.min_sim)
        if v.reverse_k:
            s, nb = knn(q, p, v.reverse_k, cfg, res)                # pool -> S1
            add(nb.ravel(), np.arange(len(p)).repeat(nb.shape[1]), s.ravel(), j, v.min_sim)
        print(f"    {v.name}: encode {t1 - t0:.0f}s  search {time.time() - t1:.0f}s", flush=True)

    # Union over views: one row per pair, one similarity column per view (0 = not found by that view).
    uniq, inv = np.unique(np.concatenate(keys), return_inverse=True)
    out = np.zeros((len(uniq), len(views)), np.float32)
    np.maximum.at(out, (inv, np.concatenate(view_ids)), np.concatenate(sims))
    return uniq // n_pool, uniq % n_pool, out


def ann_candidates(s1: pl.DataFrame, pool: pl.DataFrame, views: list[View] = VIEWS,
                   cfg: AnnConfig = AnnConfig()) -> pl.DataFrame:
    """Candidates as (s1_row, pool_row, sim_<view>...) -- row numbers into `s1` / `pool`, sorted by s1_row.

    Countries are taken from the data (open set), never hard-coded.
    """
    res = faiss.StandardGpuResources()
    s1_country, pool_country = s1["country"].to_numpy(), pool["country"].to_numpy()
    parts = []
    for country in np.unique(s1_country):
        s1_rows, pool_rows = np.flatnonzero(s1_country == country), np.flatnonzero(pool_country == country)
        if len(pool_rows) == 0:
            continue
        print(f"  {country}: {len(s1_rows):,} S1 vs {len(pool_rows):,} S2/S3", flush=True)
        qi, pi, sims = search_country(s1[s1_rows], pool[pool_rows], views, cfg, res)
        parts.append(pl.DataFrame({"s1_row": s1_rows[qi].astype(np.uint32), "pool_row": pool_rows[pi].astype(np.uint32),
                                   **{f"sim_{v.name}": sims[:, j] for j, v in enumerate(views)}}))
    return pl.concat(parts).sort("s1_row")


def with_ids(cands: pl.DataFrame, s1: pl.DataFrame, pool: pl.DataFrame) -> pl.DataFrame:
    """Add the string entity IDs (source1_entity_id, entity_id) for the row numbers."""
    return cands.with_columns(s1["entity_id"].gather(cands["s1_row"]).alias(S1_KEY),
                              pool["entity_id"].gather(cands["pool_row"]).alias(CAND_KEY))

Overwriting src/ann_blocking.py


In [21]:
import sys

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler

SRC = Path("src").resolve()
DATA = "/kaggle/input/<dataset>/student_resource/dataset"
ROOT = Path("/kaggle/working") if ON_KAGGLE else Path("../..").resolve()
CACHE, OUT = ROOT / "cache", ROOT / "output"
sys.path.insert(0, str(SRC))
from ann_blocking import AnnConfig, ann_candidates, with_ids
from candidates import CAND_KEY as CAND, S1_KEY as S1, load_split
from eval_blocking import evaluate as eval_blocking

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PY = sys.executable
BF16 = DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8  # Ampere+; Kaggle T4/P100 train in fp32
ANN = AnnConfig(dim=256)  # dim=384 recovers ~0.5pt blocking recall; needs a >= 12 GB GPU on the full test set
SAMPLE = 0.3              # share of the training set used (0.5 needs ~40 GB RAM; 0.1 fits a 16 GB machine)
VAL_SHARE = 0.4           # share of the sample held out for validation (0.4 x 0.5 = 20% of train)
TAG = f"dev{round(SAMPLE * 100)}"
torch.manual_seed(0)
print(DEVICE, torch.cuda.get_device_name() if DEVICE == "cuda" else "")

cuda Tesla T4


## 1. Normalise + dev sample
Both steps are cached; they only run when their parquet files are missing.
The dev sample is `SAMPLE` of the training S1 entities with all their true matches, plus the same share of decoys.

In [22]:
!{PY} {SRC}/prepare.py --data-dir {DATA} --cache-dir {CACHE}
if not (CACHE / f"{TAG}_links.parquet").exists():
    !{PY} {SRC}/sample.py --data-dir {DATA} --cache-dir {CACHE} --frac {SAMPLE}

dev30: S1 662,483  pool 3,097,499  true pairs 2,293,230
shape: (2, 2)
┌─────────┬────────┐
│ country ┆ len    │
│ ---     ┆ ---    │
│ str     ┆ u32    │
╞═════════╪════════╡
│ India   ┆ 265401 │
│ US      ┆ 397082 │
└─────────┴────────┘ shape: (2, 2)
┌─────────┬─────────┐
│ country ┆ len     │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ India   ┆ 1242264 │
│ US      ┆ 1855235 │
└─────────┴─────────┘


## 2. Blocking (GPU)
Per country and per view (name / address / name+address), every record becomes a 256-d count-sketch of its
char-3-gram TF-IDF vector (computed on the GPU), and FAISS finds each S1 record's top-K neighbours (see `src/ann_blocking.py`).
Candidates are row numbers into the S1 / pool frames, sorted by S1.
`eval_blocking` reports pair recall and the ceiling F0.5 a perfect matcher would reach on these candidates.

In [23]:
s1_dev, pool_dev = load_split(CACHE, TAG)
links = pl.read_parquet(CACHE / f"{TAG}_links.parquet")

cand_path = CACHE / f"{TAG}_ann{ANN.dim}.parquet"
if cand_path.exists():
    cands_dev = pl.read_parquet(cand_path)
else:
    cands_dev = ann_candidates(s1_dev, pool_dev, cfg=ANN)
    cands_dev.write_parquet(cand_path)
eval_blocking(with_ids(cands_dev, s1_dev, pool_dev), links, s1_dev)

  India: 265,401 S1 vs 1,242,264 S2/S3
    name: encode 3s  search 20s
    addr: encode 2s  search 29s
    full: encode 3s  search 59s
  US: 397,082 S1 vs 1,855,235 S2/S3
    name: encode 3s  search 43s
    addr: encode 4s  search 35s
    full: encode 4s  search 105s
pair recall: 0.9720  (2,228,946 / 2,293,230)
ceiling F0.5: 0.9903   cands per S1: 85.4 (max 1483)
shape: (2, 3)
┌─────────┬─────────────┬─────────────┐
│ country ┆ ceiling_f05 ┆ pair_recall │
│ ---     ┆ ---         ┆ ---         │
│ str     ┆ f64         ┆ f64         │
╞═════════╪═════════════╪═════════════╡
│ India   ┆ 0.982742    ┆ 0.952431    │
│ US      ┆ 0.995317    ┆ 0.985059    │
└─────────┴─────────────┴─────────────┘
  sim_name   @1=0.174  @5=0.535  @10=0.615  @20=0.671  @30=0.698
  sim_addr   @1=0.246  @5=0.741  @10=0.801  @20=0.818  @30=0.823
  sim_full   @1=0.263  @5=0.853  @10=0.908  @20=0.916  @30=0.920


## 3. Pair features
- **group**: blocking similarities, plus each candidate's gap to / rank against the best candidate of the same S1
- **string**: rapidfuzz scores on full name, core name (legal suffixes dropped) and address
- **token**: word Jaccard on core names, house/PIN-number Jaccard on addresses, source and script flags

No country feature, so unseen countries (France) go through the same model.
Computed in chunks of whole S1 groups, gathering text by row number (no joins).

In [24]:
SIMS = ["sim_name", "sim_addr", "sim_full"]
GROUP = SIMS + [f"{s}_gap" for s in SIMS] + [f"{s}_rank" for s in SIMS] + ["n_cands"]
FUZZ = {"ratio": fuzz.ratio, "tsort": fuzz.token_sort_ratio, "tset": fuzz.token_set_ratio,
        "partial": fuzz.partial_ratio, "wratio": fuzz.WRatio, "jw": JaroWinkler.normalized_similarity}
SCORERS = {"name": list(FUZZ), "core": list(FUZZ),
           "addr": ["ratio", "tsort", "tset", "jw"]}  # partial / wratio are ~8x slower on long addresses


def group_features(c: pl.DataFrame) -> pl.DataFrame:
    return c.with_columns(
        *[(pl.col(s) - pl.col(s).max().over("s1_row")).alias(f"{s}_gap") for s in SIMS],
        *[pl.col(s).rank("min", descending=True).over("s1_row").log1p().alias(f"{s}_rank") for s in SIMS],
        pl.len().over("s1_row").log1p().alias("n_cands"),
    )


def string_features(df: pl.DataFrame) -> pl.DataFrame:
    out = {}
    for f, names in SCORERS.items():
        a, b = df[f"{f}_a"].to_list(), df[f"{f}_b"].to_list()
        for k in names:
            out[f"{f}_{k}"] = process.cpdist(a, b, scorer=FUZZ[k], workers=-1) / (1 if k == "jw" else 100)
    return pl.DataFrame(out)


def jaccard(a: pl.Expr, b: pl.Expr) -> pl.Expr:
    return (a.list.set_intersection(b).list.len() / a.list.set_union(b).list.len()).fill_nan(0.0)


def featurize(cands: pl.DataFrame, s1: pl.DataFrame, pool: pl.DataFrame, s1_per_chunk: int = 12_000):
    """Yield (S1, CAND, features...) frames, each covering whole S1 groups (cands is sorted by s1_row)."""
    side = lambda s: [*[pl.col(c).fill_null("").alias(f"{c}_{s}") for c in SCORERS], pl.col("is_translit").alias(f"is_translit_{s}")]
    a = s1.select(pl.col("entity_id").alias(S1), *side("a"))
    b = pool.select(pl.col("entity_id").alias(CAND), *side("b"))
    words = lambda c: pl.col(c).str.extract_all(r"\w+")
    nums = lambda c: pl.col(c).str.extract_all(r"\d+")
    len_a, len_b = pl.col("core_a").str.len_chars(), pl.col("core_b").str.len_chars()

    rows = cands["s1_row"].to_numpy()
    cuts = np.searchsorted(rows, np.arange(0, rows[-1] + s1_per_chunk + 1, s1_per_chunk))
    for lo, hi in zip(cuts[:-1], cuts[1:]):
        if lo == hi:
            continue
        part = group_features(cands[lo:hi])
        df = part.hstack(a[part["s1_row"].to_numpy()]).hstack(b[part["pool_row"].to_numpy()])
        tokens = df.select(
            S1, CAND, *GROUP,
            core_jac=jaccard(words("core_a"), words("core_b")),
            num_jac=jaccard(nums("addr_a"), nums("addr_b")),
            num_both=(nums("addr_a").list.len() > 0) & (nums("addr_b").list.len() > 0),
            addr_empty=(pl.col("addr_a") == "") | (pl.col("addr_b") == ""),
            len_ratio=pl.min_horizontal(len_a, len_b) / pl.max_horizontal(len_a, len_b, 1),
            translit=pl.col("is_translit_a").cast(pl.Int8) + pl.col("is_translit_b").cast(pl.Int8),
            from_s3=pl.col(CAND).str.starts_with("S3-"),
        )
        yield tokens.hstack(string_features(df)).with_columns(pl.exclude(S1, CAND).cast(pl.Float32))


def to_x(df: pl.DataFrame) -> torch.Tensor:
    return torch.from_numpy(df.select(FEATURES).to_numpy())

In [25]:
feats = pl.concat(featurize(cands_dev, s1_dev, pool_dev))
FEATURES = [c for c in feats.columns if c not in (S1, CAND)]
dev = (feats.join(links.with_columns(label=pl.lit(1.0)), on=[S1, CAND], how="left")
            .with_columns(pl.col("label").fill_null(0.0)))
del feats

is_val = pl.col(S1).hash(1) % 100 < round(VAL_SHARE * 100)  # held-out S1 entities
train_df, val_df = dev.filter(~is_val), dev.filter(is_val)
val_ids = s1_dev.filter(pl.col("entity_id").hash(1) % 100 < round(VAL_SHARE * 100))["entity_id"]
del dev
print(f"{len(FEATURES)} features | train {train_df.height:,} pairs  val {val_df.height:,} pairs "
      f"| positive rate {train_df['label'].mean():.3f}")

33 features | train 33,935,992 pairs  val 22,621,891 pairs | positive rate 0.039


## 4. Matcher — PyTorch MLP on GPU
The whole training set fits in GPU memory, so batches are sliced straight from device tensors.

In [26]:
class Matcher(nn.Module):
    def __init__(self, d: int, h: int = 256, p: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(d),
            nn.Linear(d, h), nn.GELU(), nn.Dropout(p),
            nn.Linear(h, h), nn.GELU(), nn.Dropout(p),
            nn.Linear(h, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train(X, y, epochs=5, bs=8192, lr=3e-3):
    X, y = X.to(DEVICE), y.to(DEVICE)
    model = Matcher(X.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * -(-len(X) // bs))
    for ep in range(epochs):
        model.train()
        total = 0.0
        for idx in torch.randperm(len(X), device=DEVICE).split(bs):
            with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=BF16):
                loss = F.binary_cross_entropy_with_logits(model(X[idx]).float(), y[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            total += loss.item() * len(idx)
        print(f"epoch {ep + 1}: loss {total / len(X):.4f}")
    return model


@torch.no_grad()
def predict(model, X, bs=65536):
    model.eval()
    return torch.cat([torch.sigmoid(model(xb.to(DEVICE)).float()).cpu() for xb in X.split(bs)]).numpy()


model = train(to_x(train_df), torch.from_numpy(train_df["label"].to_numpy().astype(np.float32)))
del train_df
torch.cuda.empty_cache()

epoch 1: loss 0.0238
epoch 2: loss 0.0097
epoch 3: loss 0.0087
epoch 4: loss 0.0081
epoch 5: loss 0.0077


## 5. Threshold for macro F0.5
Per S1: F0.5 = 1.25·TP / (0.25·|true| + |pred|); an S1 with no true matches scores 1 only when nothing is predicted.

In [27]:
def macro_f05(pred: pl.DataFrame, links: pl.DataFrame, s1_ids: pl.Series) -> float:
    per = pl.DataFrame({S1: s1_ids})
    counts = {"n_pred": pred, "n_true": links, "tp": pred.join(links, on=[S1, CAND], how="semi")}
    for name, df in counts.items():
        per = per.join(df.group_by(S1).len(name), on=S1, how="left")
    per = per.fill_null(0)
    f = (pl.when(pl.col("n_true") + pl.col("n_pred") == 0).then(1.0)
           .otherwise(1.25 * pl.col("tp") / (0.25 * pl.col("n_true") + pl.col("n_pred"))))
    return per.select(f.mean()).item()


val_df = val_df.with_columns(p=predict(model, to_x(val_df)))
scores = {t: macro_f05(val_df.filter(pl.col("p") >= t), links, val_ids) for t in np.arange(0.30, 0.96, 0.05).round(2)}
THRESH = max(scores, key=scores.get)
print("  ".join(f"{t}:{s:.4f}" for t, s in scores.items()))
print(f"best threshold {THRESH} -> val macro F0.5 {scores[THRESH]:.4f}")

torch.save({"state": model.state_dict(), "features": FEATURES, "threshold": float(THRESH)}, CACHE / "matcher.pt")
del val_df, cands_dev, s1_dev, pool_dev  # free RAM for the test set

0.3:0.9338  0.35:0.9401  0.4:0.9448  0.45:0.9483  0.5:0.9508  0.55:0.9527  0.6:0.9542  0.65:0.9550  0.7:0.9553  0.75:0.9549  0.8:0.9538  0.85:0.9515  0.9:0.9477  0.95:0.9386
best threshold 0.7 -> val macro F0.5 0.9553


## 6. Test inference + submission
One country at a time (only that country's records are loaded): GPU blocking → features in S1 chunks → scoring.
Note: a sample of share `SAMPLE` has ~1/`SAMPLE` fewer lookalikes per S1 than the full test set, so the threshold may need to be stricter there.

In [28]:
def id_lists(pairs: pl.DataFrame, col: str) -> pl.DataFrame:
    return pairs.group_by(S1).agg(pl.col(CAND).unique().sort().str.join(",").alias(col))


def write_tsv(lists: pl.DataFrame, s1_ids: pl.Series, col: str, path: Path) -> None:
    out = pl.DataFrame({S1: s1_ids}).join(lists, on=S1, how="left").with_columns(pl.col(col).fill_null(""))
    out.write_csv(path, separator="\t", quote_style="never")


s1_scan = pl.scan_parquet(CACHE / "test_source1.parquet")
pool_scan = pl.scan_parquet([CACHE / "test_source2.parquet", CACHE / "test_source3.parquet"])
cand_lists, matches = [], []
for country in s1_scan.select(pl.col("country").unique()).collect()["country"].sort():
    s1_c = s1_scan.filter(pl.col("country") == country).collect()
    pool_c = pool_scan.filter(pl.col("country") == country).collect()
    if pool_c.height == 0:
        continue
    cands = ann_candidates(s1_c, pool_c, cfg=ANN)
    for f in featurize(cands, s1_c, pool_c):
        cand_lists.append(id_lists(f, "candidate_entity_ids"))
        matches.append(f.select(S1, CAND, p=predict(model, to_x(f))).filter(pl.col("p") >= THRESH).drop("p"))
    print(f"{country}: {cands.height:,} candidate pairs", flush=True)
    del cands, s1_c, pool_c
    torch.cuda.empty_cache()

s1_ids = s1_scan.select("entity_id").collect()["entity_id"]
matches = pl.concat(matches)
OUT.mkdir(exist_ok=True)
write_tsv(pl.concat(cand_lists), s1_ids, "candidate_entity_ids", OUT / "candidate_pairs.tsv")
write_tsv(id_lists(matches, "matched_entity_ids"), s1_ids, "matched_entity_ids", OUT / "matching_results.tsv")
print(f"{matches.height:,} matches; {matches[S1].n_unique() / len(s1_ids):.1%} of S1 entities matched")

  France: 259,452 S1 vs 1,434,993 S2/S3
    name: encode 3s  search 20s
    addr: encode 3s  search 22s
    full: encode 3s  search 66s
France: 21,743,541 candidate pairs
  India: 809,986 S1 vs 4,717,565 S2/S3


WARNING clustering 300000 points to 8687 centroids: please provide at least 338793 training points


    name: encode 9s  search 124s


WARNING clustering 300000 points to 8687 centroids: please provide at least 338793 training points


    addr: encode 11s  search 199s


WARNING clustering 300000 points to 8687 centroids: please provide at least 338793 training points


... (3 outputs left out)

WARNING clustering 300000 points to 7814 centroids: please provide at least 304746 training points


    addr: encode 8s  search 96s


WARNING clustering 300000 points to 7814 centroids: please provide at least 304746 training points


    full: encode 9s  search 285s
US: 57,003,766 candidate pairs
5,819,525 matches; 95.0% of S1 entities matched


In [29]:
!{PY} {DATA.parent}/utils/validate_submission.py -m {OUT}/matching_results.tsv -c {OUT}/candidate_pairs.tsv -t {DATA}/test

/bin/bash: line 1: {PY}: command not found


In [30]:
!cd /kaggle/working && zip -q -j results.zip output/matching_results.tsv output/candidate_pairs.tsv cache/matcher.pt && ls -lh results.zip output/

-rw-r--r-- 1 root root 811M Sep 25 15:53 results.zip

output/:
total 1.9G
-rw-r--r-- 1 root root 1.9G Sep 25 15:48 candidate_pairs.tsv
-rw-r--r-- 1 root root  93M Sep 25 15:48 matching_results.tsv


In [31]:
from IPython.display import FileLink, display

In [33]:
!cp /kaggle/working/output/matching_results.tsv /kaggle/working/
from IPython.display import FileLink, display
display(FileLink("matching_results.tsv"), FileLink("results.zip"))

/kaggle/working/matching_results.tsv

/kaggle/working/results.zip